# Phase 31 -- Claude Haiku 4.5, judged by a non-Anthropic model

Follow-up to `scripts/29...ipynb`, which tested `gemini-2.5-flash` and `gpt-4o` (both
with the compact prompt) against production and found neither cleared the pre-committed
bar. The n=6 diagnostic in `scripts/28...ipynb` had shown `claude-haiku-4.5` as the
strongest performer of everything tried, including the cleanest unprompted handling of
the q05 two-laws case -- but it was excluded from the full 60-question run specifically
because the project's judge is `anthropic/claude-sonnet-5`, same vendor as Haiku.

This notebook removes that confound: **the judge for this comparison is `openai/gpt-4o`,
not Sonnet 5.** Consequence, stated up front: these judge scores are **not on the same
scale as every other number in this project's history** (Phase 20 onward all used
Sonnet 5) -- `gpt-4o` may simply score more or less generously than Sonnet 5 across the
board. That is exactly why **production's answers are re-judged by `gpt-4o` too**,
in this same run, rather than reusing its Sonnet-5 scores from `scripts/29...ipynb` --
the comparison stays paired and internally valid even though it can't be read against
past absolute numbers.

**Efficiency: production's answers are reused, not regenerated.** An answer doesn't
change based on who grades it, so the 60 `gpt-4o-mini` (original prompt) answers from
`eval/rereview_generations.csv` are reused as-is. Only Haiku's 60 answers are newly
generated. Context is re-retrieved fresh (Pinecone is approximate-NN -- Phase 19 noted
run-to-run drift of a few recall points -- so this is not guaranteed byte-identical to
the context `rereview_generations.csv`'s production answers were originally generated
against; a small, disclosed risk rather than a hidden one) and saved to disk this time
so the run is exactly reproducible if re-judged again later.

**Decision rule, same as Phase 30 (unchanged rather than re-litigated):** adopt only if
Haiku beats production on both mean faithfulness and mean relevancy, paired, ties
excluded, by a real margin.


In [1]:
import os
import sys
import json as _json
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

from openai import OpenAI
from src.generation import (OpenAICompatibleGenerator, PRODUCTION_API_KEY_ENV, PRODUCTION_BASE_URL,
                             extract_cited_articles, grounded_article_numbers)
from src.retrieval import Retriever
from evallib import make_judge

EVAL_DIR = PROJECT / "eval"
CONTEXT_CACHE = EVAL_DIR / "haiku_rereview_context.json"


## 1. Reload the compact prompt (same string as `scripts/29...ipynb`, not re-derived)

In [2]:
from src.generation import CONTEXT_SYSTEM_PROMPT, SYSTEM_PROMPT

COMPACT_SYSTEM_PROMPT = """You are a legal information assistant for Jordanian labour and social security law (the Labour Law 1996, the Professional Work Law 2019, and the Social Security Law). Answer using only the provided excerpts.

Grounding
1. Every statement must come from the documents -- no outside knowledge.
2. If the documents don't answer the question, say so plainly rather than filling the gap.
3. Cite by law and article number (e.g. "Labour Law 1996, Article 61"), only articles you were given, under the law each document is actually labelled with.

Coverage
4. Read every document before answering. More than one article often bears on the question -- address each one that does, with its own citation, not just the first.
5. Within a cited article, cover every lettered paragraph that adds a condition, exception, limit, minimum, or further step.
6. State each provision on its own terms. Don't explain how two provisions combine or override each other unless a document says so -- give each separately, side by side.

Precision
7. Name the specific paragraph that applies ("Article 65(a)"), never merging two paragraphs into one statement.
8. Reproduce durations, percentages, amounts, ages and deadlines exactly as given, attached to the specific obligation they belong to.
9. State any qualification together with the rule it qualifies. Never present a conditional rule as unconditional or widen it beyond the persons/cases named.

Usefulness
10. Answer the person's actual situation first, in a sentence or two, then give supporting rules and citations. Don't open by restating the article.
11. If the answer depends on a fact the person hasn't given (age, years of service, marital status, a date, an amount, which law covers them, or sex where a provision sets separate figures) -- ask for it in one short question rather than guessing or covering every case. Never silently default to one case (e.g. male figures) when a provision states both. If the documents give a short, complete set of conditions instead, state them directly. Judge the current question on its own terms even if an earlier turn answered without the fact. Exception: if a computed block already gives both cases (rule 20), presenting both IS the answer -- don't also ask.
12. Reply in the question's language. Be plain and unpadded, but never drop a relevant provision to stay short.

Judgment questions
13. For a personal-decision question ("should I...", "is it worth it..."), lay out the documents' benefits and obligations/costs neutrally, side by side, so the person can weigh it. Don't recommend, and don't reason beyond what the documents state.

Totals and calculations
14. For a total, combined rate, or amount worked out from a stated figure, account for EVERY contributing provision, not just the most prominent one -- contributions especially span several articles, one per insurance type.
15. Convert any rate the documents write in words to a numeral before calculating (using the conversion aid) -- it counts the same as a digit rate.
16. Show each component on its own line with its article and insurance type (never drop the name). Then the total. Keep employer and employee amounts separate; don't combine them unless asked for the combined total.
17. Apply any stated ceiling or floor, and say plainly when it doesn't affect the figure.
18. Adding up rates the documents state outright is arithmetic, not the inference rule 6 forbids -- rule 6 still governs everything else. If the total depends on which category the person falls into, say which case you calculated and name the others.
19. A computed block's arithmetic is already done and checked -- use its rates, amounts and totals exactly, without re-deriving, re-rounding, or contradicting them. It already satisfies rules 14-17; you still cite each rate's article and add anything the documents say that the block doesn't cover.
20. Follow a block's sex-differentiated condition exactly: one figure means sex is known (state only that one); both figures mean it isn't (present both, labelled) -- this has already been determined, not something to re-decide.

Multiple laws, and who a law covers
21. If a document states a condition that decides which of two provisions applies (most often, coverage by a different law), lead with that condition -- this is reading the text, not the inference rule 6 forbids. When two laws address the same event, never merge them into one undifferentiated answer: attribute each obligation to the party the text names, and never describe one law's institution's payment (e.g. Social Security) as required from the other's party (e.g. the employer).
22. Labour Law 1996 excludes government/municipal employees and domestic/agricultural workers (separate rules apply). Professional Work Law 2019 only covers professions the Council of Ministers has included. Treat a retrieved document's own coverage conditions the same way. Mention whichever is relevant briefly, once -- not every turn, and not as a second full answer; if these documents don't cover the person's situation, say so per rule 2.

Close with one short line: general information, not legal advice, consult a licensed lawyer about their own case."""

CONTEXT_SUFFIX = CONTEXT_SYSTEM_PROMPT[len(SYSTEM_PROMPT):]
COMPACT_CONTEXT_SYSTEM_PROMPT = COMPACT_SYSTEM_PROMPT + CONTEXT_SUFFIX
print("compact prompt ready:", len(COMPACT_SYSTEM_PROMPT), "chars")


compact prompt ready: 5189 chars


## 2. Context -- retrieve fresh, cache to disk

In [3]:
ev60 = pd.read_json(EVAL_DIR / "eval_v3.jsonl", lines=True)
assert len(ev60) == 60

retriever = Retriever()
context_by_qid = {}
retr_rows = []
for q in ev60.itertuples(index=False):
    chunks = retriever.retrieve(q.question_text, top_k=5)
    context_by_qid[q.question_id] = chunks
    gold = set(q.answer_article_numbers)
    retr_rows.append({"question_id": q.question_id,
                       "full_coverage": gold.issubset({c["article_number"] for c in chunks})})

with open(CONTEXT_CACHE, "w", encoding="utf-8") as f:
    _json.dump(context_by_qid, f, ensure_ascii=False)

retr_df = pd.DataFrame(retr_rows)
print(f"full coverage: {retr_df.full_coverage.sum()}/{len(retr_df)}  (cf. Phase 30's 58/60 -- Pinecone drift check)")


full coverage: 58/60  (cf. Phase 30's 58/60 -- Pinecone drift check)


## 3. Generate -- Claude Haiku 4.5 only (production reused from `rereview_generations.csv`)

In [4]:
HAIKU_GEN_PATH = EVAL_DIR / "haiku_rereview_generations.csv"

prod_gen = pd.read_csv(EVAL_DIR / "rereview_generations.csv", encoding="utf-8-sig")
prod_gen = prod_gen[prod_gen.arm == "gpt-4o-mini (production)"].copy()
assert len(prod_gen) == 60, f"expected 60 reused production rows, got {len(prod_gen)}"

haiku_rows = []
if HAIKU_GEN_PATH.exists():
    haiku_rows = pd.read_csv(HAIKU_GEN_PATH, encoding="utf-8-sig").to_dict("records")
done = {r["question_id"] for r in haiku_rows}
print(f"{len(done)} already generated")

# max_tokens=4000: fixes the 402 hit mid-run -- OpenRouter reserves credit against Haiku's
# unset ceiling (64,000 tokens/call), not actual usage (every answer so far is well under
# 600 tokens). See the max_tokens note added to OpenAICompatibleGenerator in src/generation.py.
haiku_gen = OpenAICompatibleGenerator(model_id="anthropic/claude-haiku-4.5",
                                       api_key_env=PRODUCTION_API_KEY_ENV, base_url=PRODUCTION_BASE_URL,
                                       system_prompt=COMPACT_CONTEXT_SYSTEM_PROMPT, max_tokens=4000)

for q in ev60.itertuples(index=False):
    if q.question_id in done:
        continue
    chunks = context_by_qid[q.question_id]
    res = haiku_gen.generate(q.question_text, chunks)
    haiku_rows.append({
        "arm": "claude-haiku-4.5 (compact)", "question_id": q.question_id,
        "question_text": q.question_text, "question_language": q.question_language,
        "answer": res["answer"], "citations": res.get("citations"), "calculation": res.get("calculation"),
    })
    pd.DataFrame(haiku_rows).to_csv(HAIKU_GEN_PATH, index=False, encoding="utf-8-sig")
    print(f"  claude-haiku-4.5 / {q.question_id}", end="\r")

haiku_gen_df = pd.DataFrame(haiku_rows)
print(f"\n{len(haiku_gen_df)} Haiku generations (expect 60)")

gen_df = pd.concat([prod_gen, haiku_gen_df], ignore_index=True)
print(f"combined: {len(gen_df)} rows")


29 already generated


  claude-haiku-4.5 / q60
60 Haiku generations (expect 60)
combined: 120 rows


## 4. Regex proxy (same functions, both arms)

In [5]:
gold_by_qid = {q.question_id: set(q.answer_article_numbers) for q in ev60.itertuples(index=False)}

proxy_rows = []
for r in gen_df.itertuples(index=False):
    cited = extract_cited_articles(r.answer)
    gold = gold_by_qid[r.question_id]
    grounded = grounded_article_numbers(context_by_qid[r.question_id])
    proxy_rows.append({
        "arm": r.arm, "question_id": r.question_id,
        "correct_article_cited": bool(cited & gold),
        "all_correct_articles_cited": gold.issubset(cited),
        "has_fabrication": bool(cited - grounded),
    })

proxy_df = pd.DataFrame(proxy_rows)
proxy_df.groupby("arm")[["correct_article_cited", "all_correct_articles_cited", "has_fabrication"]].agg(
    lambda s: f"{s.sum()}/{len(s)}")


,correct_article_cited,all_correct_articles_cited,has_fabrication
arm,,,
claude-haiku-4.5 (compact),60/60,58/60,0/60
gpt-4o-mini (production),58/60,55/60,1/60


## 5. Judge -- `gpt-4o`, NOT Sonnet 5, both arms freshly scored (120 calls)

In [6]:
JUDGE_PATH = EVAL_DIR / "haiku_rereview_judge.csv"
JUDGE_MAX_TOKENS = 2000
NONANTHROPIC_JUDGE_MODEL = "openai/gpt-4o"

judge_client = OpenAI(api_key=os.environ[PRODUCTION_API_KEY_ENV], base_url=PRODUCTION_BASE_URL)
judge_one = make_judge(judge_client, model=NONANTHROPIC_JUDGE_MODEL, max_tokens=JUDGE_MAX_TOKENS)

judge_rows = []
if JUDGE_PATH.exists():
    judge_rows = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig").to_dict("records")
jdone = {(r["arm"], r["question_id"]) for r in judge_rows}
print(f"{len(jdone)} already judged, {len(gen_df) - len(jdone)} to go")

for r in gen_df.itertuples(index=False):
    if (r.arm, r.question_id) in jdone:
        continue
    verdict = judge_one(r.question_text, context_by_qid[r.question_id], r.answer)
    judge_rows.append({"arm": r.arm, "question_id": r.question_id, **verdict})
    pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print(f"  judged {len(judge_rows)}/{len(gen_df)}", end="\r")

judge_df = pd.DataFrame(judge_rows)
print(f"\n{len(judge_df)} judged")


0 already judged, 120 to go


  judged 120/120
120 judged


## 6. Results

In [7]:
print("=" * 72)
print(f"RETRIEVAL  (full coverage {retr_df.full_coverage.sum()}/{len(retr_df)})")
print("=" * 72)

print()
print("=" * 72)
print("GENERATION -- regex proxy")
print("=" * 72)
print(proxy_df.groupby("arm")[["correct_article_cited", "all_correct_articles_cited", "has_fabrication"]]
      .agg(lambda s: f"{s.sum()}/{len(s)}"))

print()
print("=" * 72)
print(f"GENERATION -- LLM judge ({NONANTHROPIC_JUDGE_MODEL}, 1-5) -- NOT comparable to Sonnet-5-judged numbers")
print("=" * 72)
print(judge_df.groupby("arm")[["faithfulness_score", "relevancy_score"]].agg(["mean", "median", "min"]))

gen_df["chars"] = gen_df.answer.str.len()
print()
print("answer length (chars):")
print(gen_df.groupby("arm").chars.agg(["mean", "median", "max"]))


RETRIEVAL  (full coverage 58/60)

GENERATION -- regex proxy
                           correct_article_cited all_correct_articles_cited  \
arm                                                                           
claude-haiku-4.5 (compact)                 60/60                      58/60   
gpt-4o-mini (production)                   58/60                      55/60   

                           has_fabrication  
arm                                         
claude-haiku-4.5 (compact)            0/60  
gpt-4o-mini (production)              1/60  

GENERATION -- LLM judge (openai/gpt-4o, 1-5) -- NOT comparable to Sonnet-5-judged numbers
                           faithfulness_score            relevancy_score  \
                                         mean median min            mean   
arm                                                                        
claude-haiku-4.5 (compact)           4.900000    5.0   3        4.966667   
gpt-4o-mini (production)             4.866667   

In [8]:
from math import comb


def binom_two_sided_p(k, n, p=0.5):
    if n == 0:
        return float("nan")
    pmf = [comb(n, i) * p**i * (1 - p)**(n - i) for i in range(n + 1)]
    pk = pmf[k]
    return min(1.0, sum(x for x in pmf if x <= pk + 1e-12))


baseline = "gpt-4o-mini (production)"
prod_scores = judge_df[judge_df.arm == baseline].set_index("question_id")
cand = judge_df[judge_df.arm == "claude-haiku-4.5 (compact)"].set_index("question_id")

print("Paired vs production, ties excluded (sign test):")
for metric in ["faithfulness_score", "relevancy_score"]:
    diff = cand[metric] - prod_scores.loc[cand.index, metric]
    wins = int((diff > 0).sum())
    losses = int((diff < 0).sum())
    n = wins + losses
    p = binom_two_sided_p(wins, n, 0.5) if n else float("nan")
    print(f"  {metric:20s} {wins}-{losses} (ties={int((diff==0).sum())})  p={p:.3f}  mean diff={diff.mean():+.3f}")


Paired vs production, ties excluded (sign test):
  faithfulness_score   5-4 (ties=51)  p=1.000  mean diff=+0.033
  relevancy_score      3-1 (ties=56)  p=0.625  mean diff=+0.033


## Verdict

**Against the pre-committed rule (beat production on both metrics, paired, by a real margin): it does not fire. Nothing adopted. But the reason it doesn't fire is a ceiling effect, not evidence of no difference -- and a separate, objective metric shows Haiku's cleanest result of the whole investigation.**

**Judge (`gpt-4o`), paired vs production:** faithfulness 5-4 (p=1.000, mean diff +0.033), relevancy 3-1 (p=0.625, mean diff +0.033). Technically Haiku "wins" both counts, but **51 of 60 questions tied on faithfulness, 56 of 60 tied on relevancy** -- there is almost nothing for a paired test to find a margin in. This traces to `gpt-4o` being a substantially more generous judge than Sonnet 5: production's own mean faithfulness came out **4.867** here vs **4.700** under Sonnet 5 in `scripts/29...ipynb` -- same answers, same questions, different judge, a full 0.17 higher. The notebook's own stated caveat (these scores aren't on Sonnet-5's scale) turned out to matter more than expected: it's not just an offset, it's a ceiling effect that erases most of the discriminating power a paired test needs.

**The regex proxy, unaffected by judge choice, is unambiguous: this is the best citation performance of every model tested in this entire investigation (Phases 29-31).** `correct_article_cited` 60/60 (matches gemini's best from Phase 30, beats gpt-4o and production's 58/60), `all_correct_articles_cited` 58/60 (beats every other arm's 55/60), **`has_fabrication` 0/60 -- the only arm across three full-eval-set runs with zero flagged fabrications**, including the known false-positive pattern that has shown up somewhere in every other arm tested.

**Not flawless, read directly rather than just touting the good numbers:** of Haiku's 4 faithfulness losses, two are real, specific errors, not just verbosity -- **q27 states the female pension-continuation age as 50 instead of the source's 55** (a wrong number, the exact failure class this project's calculators exist to prevent, though this is prose not a calculator-block question); **q21 adds an inference about "gross fault" governing personal claims against a third party that the judge found unsupported by the retrieved text** -- an ungrounded elaboration, the shape of error rule 6 exists to forbid. Both are small (n=2 of 60) but real, and worth weighing against the strong proxy numbers rather than only reporting the wins.

**Answer length:** 1,159 mean chars vs production's 804 (44% longer) -- verbose like gemini's Phase 30 result but far less extreme (gemini was 1,586, ~2x), and unlike gemini it did **not** cost Haiku a relevancy penalty here (only 1 of 60 questions lost on relevancy, and the judge's own reason for that loss was padding-shaped, not the general pattern).

**Overall reading:** this is the most encouraging candidate result across three rounds of testing (Phases 29, 30, 31) -- best proxy numbers, no truncation bug, only mild verbosity, and only two small real content errors surfaced under judge review. It still doesn't meet the bar this project set for itself before looking, and the judge comparison specifically needs a fairer instrument (a judge that doesn't ceiling out at 5/5 on 85% of questions) before it can really be trusted to confirm or deny what the proxy numbers suggest. **Not adopted. `src/generation.py` unchanged.**

**A genuine infrastructure fix did ship from this round, independent of the model decision:** `OpenAICompatibleGenerator` in `src/generation.py` now accepts an optional `max_tokens`, closing the same unset-ceiling failure mode Phase 20 found in the judge, now confirmed to also hit generation calls on models with a very high default ceiling (Haiku's is 64,000/call). Production (`gpt-4o`/`gpt-4o-mini`) has never hit this and is left with `max_tokens=None`, unchanged.
